# INC-005 Heart Rate Silver and Gold Review

This notebook inspects an existing pipeline run against the INC-005 contract (`specs/INC-005.md`): the typed `silver_heart_rate_minute` table, eleven heart-rate quality rules (range, stuck-sensor runs, `Z` timestamp conversion, duplicates) and the six heart-rate columns of `gold_clinical_daily`. The supplied database is opened read-only. The idempotency section reruns the pipeline against an isolated copy. Raw inputs are read only.

Generate the run first with `python -m wearable_pipeline run --data-dir data --output-dir output/runs/inc005`. Set `INC005_RUN_DIR` if the output directory differs, `INC005_DATA_DIR` to override `data/`, and `INC005_PREVIOUS_RUN_DIR` (default `output/runs/inc004`, an INC-004 run) to compare the pre-existing gold columns.

## Review Sections
1. Environment, paths and display helpers.
2. Row counts: gold, silver heart rate and quality results.
3. Silver schema, grain and duplicate-minute flags.
4. Heart-rate rule counts against the profiling findings (M-03, M-04, M-05, M-07, M-09) and the rule-without-rejects behavior.
5. Range, `Z` conversion and stuck-run flags recomputed independently from silver.
6. Gold daily values recomputed independently from silver.
7. Gold schema and key, and example records per situation.
8. Gaps, duplicate days and stuck-run participants named in the profiling findings.
9. Existing gold columns versus a previous run (optional).
10. Rerun on an isolated copy: identical tables and report; raw data unchanged.
11. Acceptance summary with PASS/FAIL/NOT RUN.

Aggregates are displayed, plus a few explicitly selected example gold and silver records (heart-rate columns only); reject raw-value payloads are not printed.

In [1]:
import hashlib
import html
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'wearable_pipeline').is_dir()
)
DATA_DIR = Path(os.environ.get('INC005_DATA_DIR', ROOT / 'data')).resolve()
RUN_DIR = Path(os.environ.get('INC005_RUN_DIR', ROOT / 'output' / 'runs' / 'inc005')).resolve()
PREVIOUS_RUN_DIR = Path(
    os.environ.get('INC005_PREVIOUS_RUN_DIR', ROOT / 'output' / 'runs' / 'inc004')
).resolve()
DATABASE_PATH = RUN_DIR / 'wearable.duckdb'
REPORT_PATH = RUN_DIR / 'quality' / 'report.md'

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f'DuckDB file not found: {DATABASE_PATH}. Set INC005_RUN_DIR.')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Raw data directory not found: {DATA_DIR}')

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = ''.join('<th>' + html.escape(str(column)) + '</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join('<td>' + html.escape(str(value)) + '</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<h4>' + html.escape(title) + '</h4><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body + '</tbody></table>'
    ))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


def status(ok):
    return 'PASS' if ok else 'FAIL'


report_text = REPORT_PATH.read_text(encoding='utf-8')
run_id = next(
    (line.split(': ', 1)[1] for line in report_text.splitlines() if line.startswith('Run id: ')),
    None,
)
print(f'Database: {DATABASE_PATH}')
print(f'Data directory: {DATA_DIR}')
print(f'Run ID: {run_id}')
print(f'DuckDB version: {duckdb.__version__}')

Database: /home/michaelmadeira/wearable-health-data-pipeline/output/runs/inc005/wearable.duckdb
Data directory: /home/michaelmadeira/wearable-health-data-pipeline/data
Run ID: a826c8397cf5ad8e
DuckDB version: 1.5.6


In [2]:
# Section 2: row counts against the profiled dataset.
# 2,014,950 = 42 x 40,320 + 5 x 40,326 + 3 x 39,960 source records; 41 = 30 earlier rules + 11 heart-rate rules.
expected_counts = {
    'gold_clinical_daily': 1400,
    'silver_heart_rate_minute': 2014950,
    'quality_results': 41,
}
observed_counts = {
    name: connection.execute('SELECT COUNT(*) FROM ' + name).fetchone()[0]
    for name in expected_counts
}
render_table(
    'Expected versus observed row counts',
    ['table', 'expected', 'observed', 'status'],
    [(n, e, observed_counts[n], status(observed_counts[n] == e)) for n, e in expected_counts.items()],
)
acceptance['Row counts match the profiled dataset (no heart-rate record dropped)'] = all(
    observed_counts[n] == e for n, e in expected_counts.items()
)

table,expected,observed,status
gold_clinical_daily,1400,1400,PASS
silver_heart_rate_minute,2014950,2014950,PASS
quality_results,41,41,PASS


In [3]:
# Section 3: silver schema, grain and duplicate-minute flags.
show_query(
    'Silver heart-rate schema',
    '''SELECT ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'silver_heart_rate_minute' ORDER BY ordinal_position''',
)
grain = connection.execute(
    '''SELECT COUNT(*) - COUNT(DISTINCT (participant_id, minute_ts)),
              COUNT(*) FILTER (WHERE local_date IS NULL),
              COUNT(*) FILTER (WHERE is_orphan OR is_out_of_window),
              COUNT(*) FILTER (WHERE is_duplicate_minute)
       FROM silver_heart_rate_minute'''
).fetchone()
unflagged_duplicates = connection.execute(
    '''SELECT COUNT(*) FROM silver_heart_rate_minute
       WHERE NOT is_duplicate_minute
         AND (participant_id, minute_ts) IN (
             SELECT participant_id, minute_ts FROM silver_heart_rate_minute
             GROUP BY ALL HAVING COUNT(*) > 1)'''
).fetchone()[0]
render_table(
    'Grain checks (silver keeps one row per source record, so a key can repeat only when flagged)',
    ['check', 'observed', 'status'],
    [
        ('Rows sharing a (participant_id, minute_ts) key with another row', grain[0], 'INFO'),
        ('Rows with a repeated key that are not flagged is_duplicate_minute', unflagged_duplicates,
         status(unflagged_duplicates == 0)),
        ('Rows with NULL local_date', grain[1], status(grain[1] == 0)),
        ('Orphan or out-of-window rows', grain[2], 'INFO'),
        ('Rows flagged is_duplicate_minute', grain[3], 'INFO'),
    ],
)
acceptance['Silver repeats a key only on flagged duplicate rows and every date parses'] = (
    unflagged_duplicates == 0 and grain[1] == 0
)

ordinal_position,column_name,data_type
1,participant_id,VARCHAR
2,local_date,DATE
3,minute_ts,TIMESTAMP
4,raw_timestamp,VARCHAR
5,heart_rate_bpm,INTEGER
6,heart_rate_bpm_normalized,INTEGER
7,is_orphan,BOOLEAN
8,is_out_of_window,BOOLEAN
9,is_duplicate_minute,BOOLEAN
10,is_out_of_range,BOOLEAN


check,observed,status
"Rows sharing a (participant_id, minute_ts) key with another row",5767,INFO
Rows with a repeated key that are not flagged is_duplicate_minute,0,PASS
Rows with NULL local_date,0,PASS
Orphan or out-of-window rows,0,INFO
Rows flagged is_duplicate_minute,11575,INFO


In [4]:
# Section 4: heart-rate rule counts versus the profiling findings.
show_query(
    'Heart-rate quality results',
    '''SELECT rule_id, handling, severity, checked_count, failed_count
       FROM quality_results WHERE source = 'heart_rate' ORDER BY rule_id''',
)
failed = dict(connection.execute(
    "SELECT rule_id, failed_count FROM quality_results WHERE source = 'heart_rate'"
).fetchall())
profiled = {
    'HR_VALUE_OUT_OF_RANGE': 4,
    'HR_STUCK_SENSOR_RUN': 360,
    'HR_TIMESTAMP_CONVERTED': 2880,
    'HR_TIMESTAMP_NOT_MINUTE_ALIGNED': 28,
}
render_table(
    'Profiled defect counts (M-03 out of range, M-04 stuck minutes, M-05 Z records, M-09 non-zero seconds)',
    ['rule_id', 'profiled', 'observed', 'status'],
    [(r, e, failed.get(r), status(failed.get(r) == e)) for r, e in profiled.items()],
)
print('HR_DUPLICATE_MINUTE_CONFLICT (M-07) observed:', failed.get('HR_DUPLICATE_MINUTE_CONFLICT'))
print('HR_DUPLICATE_MINUTE_EXACT (M-07) observed:', failed.get('HR_DUPLICATE_MINUTE_EXACT'))
unobserved = [
    'HR_ORPHAN_PARTICIPANT', 'HR_PARTICIPANT_FOLDER_MISMATCH', 'HR_TIMESTAMP_UNPARSEABLE',
    'HR_TIMESTAMP_OUT_OF_WINDOW', 'HR_VALUE_INVALID',
]
render_table(
    'Rules for defects not observed in profiling (expected 0)',
    ['rule_id', 'observed', 'status'],
    [(r, failed.get(r), status(failed.get(r) == 0)) for r in unobserved],
)

rejects = dict(connection.execute(
    "SELECT rule_id, COUNT(*) FROM quality_rejects WHERE source = 'heart_rate' GROUP BY 1"
).fetchall())
with_rejects = [r for r in failed if r != 'HR_TIMESTAMP_CONVERTED']
render_table(
    'Reject rows against failed_count (the informational conversion rule writes no rejects)',
    ['rule_id', 'failed_count', 'reject rows', 'status'],
    [(r, failed[r], rejects.get(r, 0), status(failed[r] == rejects.get(r, 0))) for r in sorted(with_rejects)]
    + [('HR_TIMESTAMP_CONVERTED', failed['HR_TIMESTAMP_CONVERTED'], rejects.get('HR_TIMESTAMP_CONVERTED', 0),
        status(rejects.get('HR_TIMESTAMP_CONVERTED', 0) == 0))],
)
acceptance['Profiled M-03, M-04, M-05 and M-09 counts reproduced (4 / 360 / 2,880 / 28)'] = all(
    failed.get(r) == e for r, e in profiled.items()
)
acceptance['Unobserved-defect heart-rate rules report zero'] = all(failed.get(r) == 0 for r in unobserved)
acceptance['Rejects match failed counts; the conversion rule writes none'] = (
    all(failed[r] == rejects.get(r, 0) for r in with_rejects) and rejects.get('HR_TIMESTAMP_CONVERTED', 0) == 0
)

rule_id,handling,severity,checked_count,failed_count
HR_DUPLICATE_MINUTE_CONFLICT,withhold_field,high,2014950,10393
HR_DUPLICATE_MINUTE_EXACT,flag,medium,2014950,1182
HR_ORPHAN_PARTICIPANT,exclude_from_output,medium,2014950,0
HR_PARTICIPANT_FOLDER_MISMATCH,exclude_from_output,medium,2014950,0
HR_STUCK_SENSOR_RUN,withhold_field,medium,2014950,360
HR_TIMESTAMP_CONVERTED,convert,low,2014950,2880
HR_TIMESTAMP_NOT_MINUTE_ALIGNED,flag,medium,2014950,28
HR_TIMESTAMP_OUT_OF_WINDOW,exclude_from_output,medium,2014950,0
HR_TIMESTAMP_UNPARSEABLE,exclude_from_output,medium,2014950,0
HR_VALUE_INVALID,withhold_field,medium,2014950,0


rule_id,profiled,observed,status
HR_VALUE_OUT_OF_RANGE,4,4,PASS
HR_STUCK_SENSOR_RUN,360,360,PASS
HR_TIMESTAMP_CONVERTED,2880,2880,PASS
HR_TIMESTAMP_NOT_MINUTE_ALIGNED,28,28,PASS


HR_DUPLICATE_MINUTE_CONFLICT (M-07) observed: 10393
HR_DUPLICATE_MINUTE_EXACT (M-07) observed: 1182


rule_id,observed,status
HR_ORPHAN_PARTICIPANT,0,PASS
HR_PARTICIPANT_FOLDER_MISMATCH,0,PASS
HR_TIMESTAMP_UNPARSEABLE,0,PASS
HR_TIMESTAMP_OUT_OF_WINDOW,0,PASS
HR_VALUE_INVALID,0,PASS


rule_id,failed_count,reject rows,status
HR_DUPLICATE_MINUTE_CONFLICT,10393,10393,PASS
HR_DUPLICATE_MINUTE_EXACT,1182,1182,PASS
HR_ORPHAN_PARTICIPANT,0,0,PASS
HR_PARTICIPANT_FOLDER_MISMATCH,0,0,PASS
HR_STUCK_SENSOR_RUN,360,360,PASS
HR_TIMESTAMP_NOT_MINUTE_ALIGNED,28,28,PASS
HR_TIMESTAMP_OUT_OF_WINDOW,0,0,PASS
HR_TIMESTAMP_UNPARSEABLE,0,0,PASS
HR_VALUE_INVALID,0,0,PASS
HR_VALUE_OUT_OF_RANGE,4,4,PASS


In [5]:
# Section 5: range, Z conversion and stuck runs recomputed from silver (independent of the pipeline code path).
range_checks = connection.execute(
    '''SELECT COUNT(*) FILTER (WHERE heart_rate_bpm NOT BETWEEN 40 AND 200) AS out_of_range_rows,
              COUNT(*) FILTER (WHERE is_out_of_range) AS flagged_rows,
              COUNT(*) FILTER (WHERE heart_rate_bpm NOT BETWEEN 40 AND 200 AND NOT is_out_of_range) AS unflagged,
              COUNT(*) FILTER (WHERE is_out_of_range AND heart_rate_bpm_normalized IS NOT NULL) AS flagged_but_kept,
              COUNT(*) FILTER (WHERE heart_rate_bpm_normalized IS NOT NULL
                               AND heart_rate_bpm_normalized <> heart_rate_bpm) AS changed_values,
              MIN(heart_rate_bpm_normalized), MAX(heart_rate_bpm_normalized)
       FROM silver_heart_rate_minute'''
).fetchone()
render_table(
    'Range rule: raw value kept, normalized value withheld outside 40-200',
    ['check', 'observed', 'status'],
    [
        ('Rows whose raw value is outside 40-200', range_checks[0], 'INFO'),
        ('Out-of-range rows not flagged', range_checks[2], status(range_checks[2] == 0)),
        ('Flagged rows that still carry a normalized value', range_checks[3], status(range_checks[3] == 0)),
        ('Normalized values that differ from the raw value', range_checks[4], status(range_checks[4] == 0)),
        ('Normalized minimum / maximum', f'{range_checks[5]} / {range_checks[6]}',
         status(range_checks[5] >= 40 and range_checks[6] <= 200)),
    ],
)

z_checks = connection.execute(
    '''SELECT COUNT(*),
              COUNT(DISTINCT participant_id),
              COUNT(*) FILTER (WHERE minute_ts <> CAST(REPLACE(raw_timestamp, 'Z', '') AS TIMESTAMP) - INTERVAL 5 HOUR),
              COUNT(*) FILTER (WHERE CAST(minute_ts AS DATE)
                               <> CAST(REPLACE(raw_timestamp, 'Z', '') AS DATE)),
              COUNT(*) FILTER (WHERE local_date <> CAST(minute_ts AS DATE))
       FROM silver_heart_rate_minute WHERE is_timestamp_converted'''
).fetchone()
render_table(
    'Z timestamps: converted to UTC-05:00, raw text kept (profiled: 2,880 records, 2 participants, 600 change date)',
    ['check', 'observed', 'status'],
    [
        ('Converted records', z_checks[0], status(z_checks[0] == 2880)),
        ('Participants', z_checks[1], status(z_checks[1] == 2)),
        ('Records where minute_ts differs from raw minus 5 hours', z_checks[2], status(z_checks[2] == 0)),
        ('Records whose local date differs from the raw UTC date', z_checks[3], status(z_checks[3] == 600)),
        ('Records where local_date differs from the date of minute_ts', z_checks[4], status(z_checks[4] == 0)),
    ],
)

# Stuck runs: consecutive minute slots with one in-range value, excluding conflicting-duplicate slots
# (they break a run); gaps-and-islands over the minute index.
runs = connection.execute(
    '''WITH slots AS (
           SELECT DISTINCT participant_id, date_trunc('minute', minute_ts) AS slot, heart_rate_bpm AS bpm
           FROM silver_heart_rate_minute
           WHERE minute_ts IS NOT NULL AND heart_rate_bpm BETWEEN 40 AND 200 AND NOT is_out_of_window
             AND (participant_id, date_trunc('minute', minute_ts)) NOT IN (
                 SELECT participant_id, date_trunc('minute', minute_ts) FROM silver_heart_rate_minute
                 WHERE quality_flags LIKE '%HR_DUPLICATE_MINUTE_CONFLICT%')),
       indexed AS (
           SELECT participant_id, slot, bpm,
                  CAST(epoch(slot) / 60 AS BIGINT) AS idx FROM slots),
       islands AS (
           SELECT participant_id, slot, bpm,
                  idx - ROW_NUMBER() OVER (PARTITION BY participant_id, bpm ORDER BY idx) AS island
           FROM indexed),
       runs AS (
           SELECT participant_id, bpm, island, MIN(slot) AS run_start, MAX(slot) AS run_end, COUNT(*) AS run_length
           FROM islands GROUP BY ALL HAVING COUNT(*) >= 60)
       SELECT participant_id, bpm, run_start, run_end, run_length FROM runs ORDER BY 1, 3'''
).fetchall()
render_table(
    'Stuck runs of at least 60 identical consecutive minutes recomputed from silver (profiled: 2 runs of 180)',
    ['participant_id', 'value', 'run start', 'run end', 'minutes'],
    runs,
)
flagged_stuck = connection.execute(
    '''SELECT COUNT(*), COUNT(*) FILTER (WHERE heart_rate_bpm_normalized IS NOT NULL),
              COUNT(*) FILTER (WHERE quality_flags NOT LIKE '%HR_STUCK_SENSOR_RUN%')
       FROM silver_heart_rate_minute WHERE is_stuck_sensor'''
).fetchone()
stuck_pairs = connection.execute(
    '''SELECT participant_id, MIN(minute_ts), MAX(minute_ts), COUNT(*)
       FROM silver_heart_rate_minute WHERE is_stuck_sensor GROUP BY 1 ORDER BY 1'''
).fetchall()
recomputed_pairs = [(r[0], r[2], r[3], r[4]) for r in runs]
flagged_pairs = [(r[0], r[1], r[2], r[3]) for r in stuck_pairs]
same_runs = recomputed_pairs == flagged_pairs
render_table(
    'Stuck-run flags versus the recomputed runs',
    ['check', 'observed', 'status'],
    [
        ('Flagged minutes', flagged_stuck[0], status(flagged_stuck[0] == 360)),
        ('Flagged minutes that still carry a normalized value', flagged_stuck[1], status(flagged_stuck[1] == 0)),
        ('Flagged rows missing the rule id in quality_flags', flagged_stuck[2], status(flagged_stuck[2] == 0)),
        ('Flagged runs equal the recomputed runs (participant, start, end, minutes)', same_runs, status(same_runs)),
    ],
)
acceptance['Out-of-range values are flagged and withheld, raw kept'] = (
    range_checks[2] == 0 and range_checks[3] == 0 and range_checks[4] == 0
)
acceptance['Z records converted to UTC-05:00 (2,880 records, 600 change date), raw text kept'] = (
    z_checks == (2880, 2, 0, 600, 0)
)
acceptance['Stuck-run flags equal an independent recomputation (whole run flagged, values withheld)'] = (
    same_runs and flagged_stuck[0] == 360 and flagged_stuck[1] == 0 and flagged_stuck[2] == 0
)

check,observed,status
Rows whose raw value is outside 40-200,4,INFO
Out-of-range rows not flagged,0,PASS
Flagged rows that still carry a normalized value,0,PASS
Normalized values that differ from the raw value,0,PASS
Normalized minimum / maximum,42 / 136,PASS


check,observed,status
Converted records,2880,PASS
Participants,2,PASS
Records where minute_ts differs from raw minus 5 hours,0,PASS
Records whose local date differs from the raw UTC date,600,PASS
Records where local_date differs from the date of minute_ts,0,PASS


participant_id,value,run start,run end,minutes
P003,60,2026-01-09 04:44:00,2026-01-09 07:43:00,180
P042,73,2026-01-24 09:16:00,2026-01-24 12:15:00,180


check,observed,status
Flagged minutes,360,PASS
Flagged minutes that still carry a normalized value,0,PASS
Flagged rows missing the rule id in quality_flags,0,PASS
"Flagged runs equal the recomputed runs (participant, start, end, minutes)",True,PASS


In [6]:
# Section 6: gold daily values recomputed independently from silver.
# One value per participant-minute slot (exact duplicates count once; conflicting duplicates and withheld
# values are NULL in the normalized column), in-scope rows only.
gold_diff = connection.execute(
    '''WITH slots AS (
           SELECT participant_id, local_date, date_trunc('minute', minute_ts) AS slot,
                  MAX(heart_rate_bpm_normalized) AS bpm
           FROM silver_heart_rate_minute
           WHERE NOT is_orphan AND NOT is_out_of_window AND heart_rate_bpm_normalized IS NOT NULL
           GROUP BY ALL),
       daily AS (
           SELECT participant_id, local_date, AVG(bpm) AS mean_bpm, MIN(bpm) AS min_bpm,
                  MAX(bpm) AS max_bpm, COUNT(*) AS valid
           FROM slots GROUP BY ALL)
       SELECT COUNT(*) FROM gold_clinical_daily g
       LEFT JOIN daily d USING (participant_id, local_date)
       WHERE (g.hr_mean_bpm IS NULL) <> (d.mean_bpm IS NULL)
          OR ABS(g.hr_mean_bpm - d.mean_bpm) > 1e-9
          OR g.hr_min_bpm IS DISTINCT FROM d.min_bpm
          OR g.hr_max_bpm IS DISTINCT FROM d.max_bpm
          OR g.hr_valid_minute_count <> COALESCE(d.valid, 0)'''
).fetchone()[0]
present_diff = connection.execute(
    '''WITH present AS (
           SELECT DISTINCT participant_id, local_date FROM silver_heart_rate_minute
           WHERE NOT is_orphan AND NOT is_out_of_window AND local_date IS NOT NULL)
       SELECT COUNT(*) FROM gold_clinical_daily g
       LEFT JOIN present p USING (participant_id, local_date)
       WHERE g.hr_present <> (p.participant_id IS NOT NULL)'''
).fetchone()[0]
silver_valid, gold_valid = connection.execute(
    '''SELECT (SELECT COUNT(*) FROM silver_heart_rate_minute WHERE heart_rate_bpm_normalized IS NOT NULL),
              (SELECT SUM(hr_valid_minute_count) FROM gold_clinical_daily)'''
).fetchone()
exact_duplicate_records = failed['HR_DUPLICATE_MINUTE_EXACT']
null_rules = connection.execute(
    '''SELECT COUNT(*) FROM gold_clinical_daily
       WHERE (hr_valid_minute_count = 0) <> (hr_mean_bpm IS NULL AND hr_min_bpm IS NULL AND hr_max_bpm IS NULL)
          OR (NOT hr_present AND hr_quality_flags IS NOT NULL)'''
).fetchone()[0]
render_table(
    'Gold heart-rate values versus an independent recomputation from silver',
    ['check', 'observed', 'status'],
    [
        ('Gold rows whose mean, min, max or valid count differ', gold_diff, status(gold_diff == 0)),
        ('Gold rows whose hr_present differs', present_diff, status(present_diff == 0)),
        ('Days with NULL aggregates but a valid minute, or the reverse; flags on a day without records', null_rules,
         status(null_rules == 0)),
        ('Silver normalized values / gold valid minutes', f'{silver_valid} / {gold_valid}', 'INFO'),
        ('Difference equals half of the exact-duplicate records (each counted once)',
         f'{silver_valid - gold_valid} versus {exact_duplicate_records // 2}',
         status(silver_valid - gold_valid == exact_duplicate_records // 2)),
    ],
)
show_query(
    'Daily aggregate ranges over days with valid minutes',
    '''SELECT MIN(hr_min_bpm) AS lowest_min, MAX(hr_max_bpm) AS highest_max,
              ROUND(MIN(hr_mean_bpm), 2) AS lowest_mean, ROUND(MAX(hr_mean_bpm), 2) AS highest_mean,
              COUNT(*) AS days
       FROM gold_clinical_daily WHERE hr_valid_minute_count > 0''',
)
acceptance['Gold heart-rate columns equal an independent recomputation from silver'] = (
    gold_diff == 0 and present_diff == 0 and null_rules == 0
)
acceptance['Exact duplicates are counted once in the valid-minute total'] = (
    silver_valid - gold_valid == exact_duplicate_records // 2
)

check,observed,status
"Gold rows whose mean, min, max or valid count differ",0,PASS
Gold rows whose hr_present differs,0,PASS
"Days with NULL aggregates but a valid minute, or the reverse; flags on a day without records",0,PASS
Silver normalized values / gold valid minutes,2004193 / 2003602,INFO
Difference equals half of the exact-duplicate records (each counted once),591 versus 591,PASS


lowest_min,highest_max,lowest_mean,highest_mean,days
42,136,54.38,104.78,1396


In [7]:
# Section 7a: gold_clinical_daily schema and primary key.
show_query(
    'Gold schema (six heart-rate columns are appended after the ten survey, six sleep and six steps columns)',
    '''SELECT ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position''',
)
show_query(
    'Gold key constraints',
    '''SELECT constraint_type, constraint_column_names
       FROM duckdb_constraints() WHERE table_name = 'gold_clinical_daily' ''',
)
gold_columns = [row[0] for row in connection.execute(
    "SELECT column_name FROM information_schema.columns "
    "WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position").fetchall()]
acceptance['Gold has 28 columns ending with the six heart-rate columns'] = (
    len(gold_columns) == 28
    and gold_columns[-6:] == ['hr_mean_bpm', 'hr_min_bpm', 'hr_max_bpm', 'hr_valid_minute_count',
                              'hr_present', 'hr_quality_flags']
)

ordinal_position,column_name,data_type
1,participant_id,VARCHAR
2,local_date,DATE
3,fatigue_score,INTEGER
4,stress_score,INTEGER
5,readiness_score,INTEGER
6,sleep_quality_score,INTEGER
7,survey_present,BOOLEAN
8,survey_row_count,INTEGER
9,survey_valid_score_count,INTEGER
10,survey_quality_flags,VARCHAR


constraint_type,constraint_column_names
PRIMARY KEY,"['participant_id', 'local_date']"
NOT NULL,['participant_id']
NOT NULL,['local_date']


In [8]:
# Section 7b: example gold records (one deterministic example per situation, heart-rate columns plus the key).
hr_columns = (
    'participant_id, local_date, hr_mean_bpm, hr_min_bpm, hr_max_bpm, hr_valid_minute_count, '
    'hr_present, hr_quality_flags'
)
situations = [
    ('Clean heart-rate day', 'hr_present AND hr_quality_flags IS NULL AND hr_valid_minute_count = 1440'),
    ('Day with converted Z records (M-05)', "hr_quality_flags LIKE '%HR_TIMESTAMP_CONVERTED%'"),
    ('Day with an out-of-range value (M-03)', "hr_quality_flags LIKE '%HR_VALUE_OUT_OF_RANGE%'"),
    ('Day with a stuck-sensor run (M-04)', "hr_quality_flags LIKE '%HR_STUCK_SENSOR_RUN%'"),
    ('Day with conflicting duplicates (M-07)', "hr_quality_flags LIKE '%HR_DUPLICATE_MINUTE_CONFLICT%'"),
    ('Day with a non-aligned timestamp (M-09)', "hr_quality_flags LIKE '%HR_TIMESTAMP_NOT_MINUTE_ALIGNED%'"),
    ('Day with partial coverage and no flag (M-08)',
     'hr_valid_minute_count BETWEEN 1 AND 1439 AND hr_quality_flags IS NULL'),
    ('Day without heart-rate records', 'NOT hr_present'),
]
for title, condition in situations:
    cursor = connection.execute(
        'SELECT ' + hr_columns + ' FROM gold_clinical_daily WHERE ' + condition
        + ' ORDER BY participant_id, local_date LIMIT 1'
    )
    rows = cursor.fetchall()
    render_table(title, [item[0] for item in cursor.description],
                 rows or [('no such day in this run',) + ('',) * (len(cursor.description) - 1)])
show_query(
    'Same example day across layers: gold row and its silver records by level (day with converted Z records)',
    '''SELECT g.participant_id, g.local_date, g.hr_valid_minute_count,
              COUNT(s.minute_ts) AS silver_records,
              COUNT(s.minute_ts) FILTER (WHERE s.is_timestamp_converted) AS converted_records,
              COUNT(s.minute_ts) FILTER (WHERE s.heart_rate_bpm_normalized IS NOT NULL) AS valid_records
       FROM gold_clinical_daily g
       JOIN silver_heart_rate_minute s USING (participant_id, local_date)
       WHERE g.hr_quality_flags LIKE '%HR_TIMESTAMP_CONVERTED%'
       GROUP BY ALL ORDER BY g.participant_id, g.local_date LIMIT 1''',
)

participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P001,2026-01-08,60.270833333333336,49,74,1440,True,None


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P018,2026-01-25,65.025,56,78,1440,True,HR_TIMESTAMP_CONVERTED


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P029,2026-01-16,60.904100069492706,52,74,1439,True,HR_VALUE_OUT_OF_RANGE


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P003,2026-01-09,65.83968253968254,57,82,1260,True,HR_STUCK_SENSOR_RUN


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P005,2026-01-10,103.7866574009729,64,136,1439,True,"HR_DUPLICATE_MINUTE_CONFLICT,HR_TIMESTAMP_NOT_MINUTE_ALIGNED"


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P005,2026-01-10,103.7866574009729,64,136,1439,True,"HR_DUPLICATE_MINUTE_CONFLICT,HR_TIMESTAMP_NOT_MINUTE_ALIGNED"


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P020,2026-01-27,73.21944444444445,65,82,1080,True,None


participant_id,local_date,hr_mean_bpm,hr_min_bpm,hr_max_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P010,2026-01-08,None,None,None,0,False,None


participant_id,local_date,hr_valid_minute_count,silver_records,converted_records,valid_records
P018,2026-01-25,1440,1440,1440,1440


In [9]:
# Section 7c: example silver records (heart-rate columns only; raw-value payloads are not printed).
silver_columns = (
    'participant_id, local_date, minute_ts, raw_timestamp, heart_rate_bpm, heart_rate_bpm_normalized, '
    'is_duplicate_minute, quality_flags'
)
silver_situations = [
    ('Converted Z record (raw text kept, local timestamp derived)', 'is_timestamp_converted'),
    ('Converted Z record that changes the local date',
     'is_timestamp_converted AND CAST(minute_ts AS DATE) <> CAST(REPLACE(raw_timestamp, \'Z\', \'\') AS DATE)'),
    ('Out-of-range record (value withheld, raw value kept)', 'is_out_of_range'),
    ('Stuck-sensor record (raw value kept, normalized value withheld)', 'is_stuck_sensor'),
    ('Conflicting duplicate record (normalized value withheld)',
     "quality_flags LIKE '%HR_DUPLICATE_MINUTE_CONFLICT%'"),
    ('Exact duplicate record (value kept, counted once in gold)',
     "quality_flags LIKE '%HR_DUPLICATE_MINUTE_EXACT%'"),
    ('Non-aligned timestamp record (seconds kept, not floored)',
     "quality_flags LIKE '%HR_TIMESTAMP_NOT_MINUTE_ALIGNED%'"),
]
for title, condition in silver_situations:
    cursor = connection.execute(
        'SELECT ' + silver_columns + ' FROM silver_heart_rate_minute WHERE ' + condition
        + ' ORDER BY participant_id, minute_ts LIMIT 2'
    )
    rows = cursor.fetchall()
    render_table(title, [item[0] for item in cursor.description],
                 rows or [('no such record in this run',) + ('',) * (len(cursor.description) - 1)])

# Full gold rows, transposed so all 28 columns fit on screen.
for title, condition in [
    ('Gold row with a stuck-sensor run, every column', "hr_quality_flags LIKE '%HR_STUCK_SENSOR_RUN%'"),
    ('Gold row without heart-rate records, every column', 'NOT hr_present'),
]:
    cursor = connection.execute(
        'SELECT * FROM gold_clinical_daily WHERE ' + condition + ' ORDER BY participant_id, local_date LIMIT 2'
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    render_table(
        title,
        ['column'] + ['example ' + str(n) for n in range(1, len(rows) + 1)] if rows else ['column'],
        [(name, *(row[i] for row in rows)) for i, name in enumerate(columns)] if rows else [('no such day in this run',)],
    )

participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P018,2026-01-25,2026-01-25 00:00:00,2026-01-25T05:00:00Z,66,66,False,HR_TIMESTAMP_CONVERTED
P018,2026-01-25,2026-01-25 00:01:00,2026-01-25T05:01:00Z,65,65,False,HR_TIMESTAMP_CONVERTED


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P018,2026-01-25,2026-01-25 19:00:00,2026-01-26T00:00:00Z,67,67,False,HR_TIMESTAMP_CONVERTED
P018,2026-01-25,2026-01-25 19:01:00,2026-01-26T00:01:00Z,65,65,False,HR_TIMESTAMP_CONVERTED


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P029,2026-01-16,2026-01-16 18:47:00,None,-10,None,False,HR_VALUE_OUT_OF_RANGE
P029,2026-01-23,2026-01-23 21:53:00,None,999,None,False,HR_VALUE_OUT_OF_RANGE


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P003,2026-01-09,2026-01-09 04:44:00,None,60,None,False,HR_STUCK_SENSOR_RUN
P003,2026-01-09,2026-01-09 04:45:00,None,60,None,False,HR_STUCK_SENSOR_RUN


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P005,2026-01-10,2026-01-10 13:55:00,None,71,None,True,HR_DUPLICATE_MINUTE_CONFLICT
P005,2026-01-10,2026-01-10 13:55:22,None,123,None,True,"HR_DUPLICATE_MINUTE_CONFLICT,HR_TIMESTAMP_NOT_MINUTE_ALIGNED"


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P010,2026-01-10,2026-01-10 00:01:00,None,66,66,True,HR_DUPLICATE_MINUTE_EXACT
P010,2026-01-10,2026-01-10 00:01:00,None,66,66,True,HR_DUPLICATE_MINUTE_EXACT


participant_id,local_date,minute_ts,raw_timestamp,heart_rate_bpm,heart_rate_bpm_normalized,is_duplicate_minute,quality_flags
P005,2026-01-10,2026-01-10 13:55:22,None,127,None,True,"HR_DUPLICATE_MINUTE_CONFLICT,HR_TIMESTAMP_NOT_MINUTE_ALIGNED"
P005,2026-01-10,2026-01-10 13:55:22,None,123,None,True,"HR_DUPLICATE_MINUTE_CONFLICT,HR_TIMESTAMP_NOT_MINUTE_ALIGNED"


column,example 1,example 2
participant_id,P003,P042
local_date,2026-01-09,2026-01-24
fatigue_score,2,3
stress_score,3,3
readiness_score,6,6
sleep_quality_score,2,3
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


column,example 1,example 2
participant_id,P010,P019
local_date,2026-01-08,2026-01-08
fatigue_score,3,3
stress_score,5,4
readiness_score,7,5
sleep_quality_score,3,2
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


In [10]:
# Section 8: gaps, duplicate days and stuck-run participants named in the profiling findings.
show_query(
    'Missing day 2026-01-08 (P010, P019, P022, P025): no records, so NULL aggregates and hr_present false',
    '''SELECT participant_id, local_date, hr_mean_bpm, hr_valid_minute_count, hr_present, hr_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-08'
       ORDER BY 1''',
)
show_query(
    'Duplicated day 2026-01-10 (same participants): conflicting minutes withheld',
    '''SELECT participant_id, local_date, hr_mean_bpm, hr_valid_minute_count, hr_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-10'
       ORDER BY 1''',
)
show_query(
    'Partial days (P020, P028, P041)',
    '''SELECT participant_id, local_date, hr_valid_minute_count, hr_mean_bpm, hr_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P020', 'P028', 'P041') AND hr_valid_minute_count BETWEEN 1 AND 1439
       ORDER BY 1, 2''',
)
show_query(
    'Stuck-run days (P003 and P042) and Z-record days (P018 and P044)',
    '''SELECT participant_id, local_date, hr_valid_minute_count, hr_quality_flags
       FROM gold_clinical_daily
       WHERE hr_quality_flags LIKE '%HR_STUCK_SENSOR_RUN%' OR (participant_id IN ('P018', 'P044')
             AND hr_quality_flags LIKE '%HR_TIMESTAMP_CONVERTED%' AND local_date <= DATE '2026-01-25')
       ORDER BY 1, 2 LIMIT 12''',
)
no_records = connection.execute(
    '''SELECT COUNT(*) FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-08'
         AND hr_mean_bpm IS NULL AND hr_valid_minute_count = 0 AND NOT hr_present'''
).fetchone()[0]
withheld = connection.execute(
    '''SELECT COUNT(*) FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-10'
         AND hr_valid_minute_count < 1440
         AND hr_quality_flags LIKE '%HR_DUPLICATE_MINUTE_CONFLICT%' '''
).fetchone()[0]
partial = connection.execute(
    '''SELECT COUNT(DISTINCT participant_id) FROM gold_clinical_daily
       WHERE participant_id IN ('P020', 'P028', 'P041') AND hr_valid_minute_count BETWEEN 1 AND 1439'''
).fetchone()[0]
acceptance['2026-01-08 gaps show NULL aggregates, count 0 and hr_present false (4 participants)'] = no_records == 4
acceptance['2026-01-10 conflicting duplicates withheld (4 participants)'] = withheld == 4
acceptance['Partial days visible as lower valid-minute counts (3 participants)'] = partial == 3

participant_id,local_date,hr_mean_bpm,hr_valid_minute_count,hr_present,hr_quality_flags
P010,2026-01-08,None,0,False,None
P019,2026-01-08,None,0,False,None
P022,2026-01-08,None,0,False,None
P025,2026-01-08,None,0,False,None


participant_id,local_date,hr_mean_bpm,hr_valid_minute_count,hr_quality_flags
P010,2026-01-10,66.45270270270271,148,"HR_DUPLICATE_MINUTE_CONFLICT,HR_DUPLICATE_MINUTE_EXACT"
P019,2026-01-10,67.92142857142858,140,"HR_DUPLICATE_MINUTE_CONFLICT,HR_DUPLICATE_MINUTE_EXACT"
P022,2026-01-10,75.06666666666666,150,"HR_DUPLICATE_MINUTE_CONFLICT,HR_DUPLICATE_MINUTE_EXACT"
P025,2026-01-10,70.74834437086092,151,"HR_DUPLICATE_MINUTE_CONFLICT,HR_DUPLICATE_MINUTE_EXACT"


participant_id,local_date,hr_valid_minute_count,hr_mean_bpm,hr_quality_flags
P020,2026-01-27,1080,73.21944444444445,None
P028,2026-01-15,1080,73.30833333333334,None
P041,2026-01-29,1080,55.61851851851852,None


participant_id,local_date,hr_valid_minute_count,hr_quality_flags
P003,2026-01-09,1260,HR_STUCK_SENSOR_RUN
P018,2026-01-25,1440,HR_TIMESTAMP_CONVERTED
P042,2026-01-24,1260,HR_STUCK_SENSOR_RUN
P044,2026-01-17,1440,HR_TIMESTAMP_CONVERTED


In [11]:
# Section 9: pre-existing gold columns versus a previous (INC-004) run, when it exists.
existing_columns = ', '.join(gold_columns[:22])
previous_db = PREVIOUS_RUN_DIR / 'wearable.duckdb'
if previous_db.is_file():
    previous = duckdb.connect(str(previous_db), read_only=True)
    try:
        before = previous.execute('SELECT ' + existing_columns + ' FROM gold_clinical_daily ORDER BY ALL').fetchall()
    finally:
        previous.close()
    after = connection.execute('SELECT ' + existing_columns + ' FROM gold_clinical_daily ORDER BY ALL').fetchall()
    identical = before == after
    render_table(
        'Existing 22 gold columns versus previous run',
        ['rows before', 'rows after', 'identical', 'status'],
        [(len(before), len(after), identical, status(identical))],
    )
    acceptance['Existing 22 gold columns identical to the previous run'] = identical
else:
    print('Previous run not found at', previous_db, ': comparison NOT RUN.')
    acceptance['Existing 22 gold columns identical to the previous run'] = None

rows before,rows after,identical,status
1400,1400,True,PASS


In [12]:
# Section 10: rerun on an isolated copy and check raw-data immutability against the run manifest.
# Tables are compared inside DuckDB (EXCEPT ALL in both directions) to avoid loading 2 million rows into Python.
connection.close()
sys.path.insert(0, str(ROOT / 'src'))
from wearable_pipeline.pipeline import run_pipeline

table_names = [
    'raw_manifest', 'silver_participants', 'silver_wellness_survey', 'silver_sleep_sessions',
    'silver_sleep_stages', 'silver_steps_minute', 'silver_heart_rate_minute', 'quality_results',
    'quality_rejects', 'gold_clinical_daily',
]
rerun_comparisons = []
with tempfile.TemporaryDirectory(prefix='inc005-idempotency-review-') as temporary_root:
    temporary_root = Path(temporary_root)
    baseline_dir = temporary_root / 'baseline'
    rerun_dir = temporary_root / 'rerun'
    for directory in (baseline_dir, rerun_dir):
        (directory / 'quality').mkdir(parents=True)
        shutil.copy2(DATABASE_PATH, directory / 'wearable.duckdb')
        shutil.copy2(REPORT_PATH, directory / 'quality' / 'report.md')

    rerun_result = run_pipeline(DATA_DIR, rerun_dir)
    compare = duckdb.connect(':memory:')
    compare.execute("ATTACH '" + str(baseline_dir / 'wearable.duckdb') + "' AS before_db (READ_ONLY)")
    compare.execute("ATTACH '" + str(rerun_dir / 'wearable.duckdb') + "' AS after_db (READ_ONLY)")
    try:
        for table_name in table_names:
            rows_before = compare.execute('SELECT COUNT(*) FROM before_db.' + table_name).fetchone()[0]
            only_before = compare.execute(
                'SELECT COUNT(*) FROM (SELECT * FROM before_db.' + table_name
                + ' EXCEPT ALL SELECT * FROM after_db.' + table_name + ')').fetchone()[0]
            only_after = compare.execute(
                'SELECT COUNT(*) FROM (SELECT * FROM after_db.' + table_name
                + ' EXCEPT ALL SELECT * FROM before_db.' + table_name + ')').fetchone()[0]
            rerun_comparisons.append((table_name, rows_before, only_before == 0 and only_after == 0))
    finally:
        compare.close()
    report_identical = (baseline_dir / 'quality' / 'report.md').read_bytes() == (
        rerun_dir / 'quality' / 'report.md'
    ).read_bytes()
    run_id_identical = rerun_result.run_id == run_id

render_table(
    'Same-input rerun comparison',
    ['table', 'rows before', 'identical after rerun', 'status'],
    [(*row, status(row[2])) for row in rerun_comparisons],
)
render_table(
    'Rerun determinism',
    ['check', 'result', 'status'],
    [
        ('Run ID unchanged', run_id_identical, status(run_id_identical)),
        ('Quality report bytes unchanged', report_identical, status(report_identical)),
    ],
)
acceptance["Rerun preserves every table's logical contents"] = all(row[2] for row in rerun_comparisons)
acceptance['Rerun preserves the deterministic run id and report'] = run_id_identical and report_identical

check_db = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = {
    row[0]: row[1]
    for row in check_db.execute('SELECT relative_path, sha256 FROM raw_manifest').fetchall()
}
check_db.close()
hash_mismatches = [
    path for path, digest in manifest.items()
    if hashlib.sha256((DATA_DIR / path).read_bytes()).hexdigest() != digest
]
print('Raw files differing from the run manifest SHA-256:', hash_mismatches[:5] or 'none')
acceptance['Raw files match the run manifest hashes'] = not hash_mismatches

table,rows before,identical after rerun,status
raw_manifest,154,True,PASS
silver_participants,50,True,PASS
silver_wellness_survey,1414,True,PASS
silver_sleep_sessions,1400,True,PASS
silver_sleep_stages,7000,True,PASS
silver_steps_minute,2014950,True,PASS
silver_heart_rate_minute,2014950,True,PASS
quality_results,41,True,PASS
quality_rejects,26368,True,PASS
gold_clinical_daily,1400,True,PASS


check,result,status
Run ID unchanged,True,PASS
Quality report bytes unchanged,True,PASS


Raw files differing from the run manifest SHA-256: none


In [13]:
# Section 11: acceptance summary.
rows = [
    (name, 'PASS' if result is True else 'FAIL' if result is False else 'NOT RUN')
    for name, result in acceptance.items()
]
rows.append(('Changed-input updates affect exactly one gold row', 'NOT RUN here; covered by the pipeline tests'))
render_table('INC-005 review summary', ['acceptance check', 'status'], rows)
print('Review is observational; only the isolated temporary database copy was rerun.')

acceptance check,status
Row counts match the profiled dataset (no heart-rate record dropped),PASS
Silver repeats a key only on flagged duplicate rows and every date parses,PASS
"Profiled M-03, M-04, M-05 and M-09 counts reproduced (4 / 360 / 2,880 / 28)",PASS
Unobserved-defect heart-rate rules report zero,PASS
Rejects match failed counts; the conversion rule writes none,PASS
"Out-of-range values are flagged and withheld, raw kept",PASS
"Z records converted to UTC-05:00 (2,880 records, 600 change date), raw text kept",PASS
"Stuck-run flags equal an independent recomputation (whole run flagged, values withheld)",PASS
Gold heart-rate columns equal an independent recomputation from silver,PASS
Exact duplicates are counted once in the valid-minute total,PASS


Review is observational; only the isolated temporary database copy was rerun.
